# RL Movie - ML-Agents Training Notebook

このノートブックは `Unity -> Build for Colab -> Colab 学習 -> Import Trained Model` の標準パイプライン用です。
`scenario_manifest.yaml` を読み、`run_id` 規約と実験メタデータを検証しながら学習を実行します。

---
## Step 0: 実験パラメータ

In [ ]:
#@title Basic Parameters { run: "auto" }
SCENARIO_NAME = "RollerBall"  #@param {type:"string"}
RUN_TAG = "baseline"  #@param {type:"string"}
RUN_ID = ""  #@param {type:"string"}
SPEC_VERSION = 1  #@param {type:"integer"}
HYPOTHESIS = "baseline environment and reward shaping"  #@param {type:"string"}
SEED = 1  #@param {type:"integer"}
BASELINE_RUN = ""  #@param {type:"string"}
MAX_STEPS = 500000  #@param {type:"integer"}
RESUME_TRAINING = False  #@param {type:"boolean"}

import os
import re
from datetime import datetime, timezone

def to_snake_case(name):
    return re.sub(r"(?<!^)(?=[A-Z])", "_", name).lower()

def build_run_id(scenario_slug, spec_version, run_tag):
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M")
    normalized_tag = re.sub(r"[^a-z0-9-]+", "-", run_tag.strip().lower()).strip("-")
    if not normalized_tag:
        raise ValueError("RUN_TAG must contain at least one lowercase letter or number")
    return f"{scenario_slug}__v{spec_version}__{normalized_tag}__{timestamp}"

SCENARIO_SLUG = to_snake_case(SCENARIO_NAME)
if not RUN_ID.strip():
    RUN_ID = build_run_id(SCENARIO_SLUG, SPEC_VERSION, RUN_TAG)

RUN_ID_PATTERN = rf"^{SCENARIO_SLUG}__v{SPEC_VERSION}__[a-z0-9]+(?:-[a-z0-9]+)*__\d{{8}}_\d{{4}}$"

DRIVE_PROJECT = "/content/drive/MyDrive/RL-Movie"
DRIVE_BUILDS = f"{DRIVE_PROJECT}/Builds"
DRIVE_RESULTS = f"{DRIVE_PROJECT}/Results"
DRIVE_MODELS = f"{DRIVE_PROJECT}/Models"
WORK_DIR = "/content/training"
BUILD_DIR = f"{WORK_DIR}/{SCENARIO_NAME}"
CONFIG_DIR = f"{BUILD_DIR}/Config"
RUN_RESULTS_DIR = f"{DRIVE_RESULTS}/{RUN_ID}"
MODEL_EXPORT_DIR = f"{DRIVE_MODELS}/{SCENARIO_NAME}/{RUN_ID}"
RUN_CONTEXT_PATH = f"{RUN_RESULTS_DIR}/run_context.json"
RUN_PROGRESS_PATH = f"{RUN_RESULTS_DIR}/progress.json"
RUN_LOG_PATH = f"{RUN_RESULTS_DIR}/training_output.log"

print(f"Scenario:      {SCENARIO_NAME}")
print(f"Scenario slug: {SCENARIO_SLUG}")
print(f"Run tag:       {RUN_TAG}")
print(f"Run ID:        {RUN_ID}")
print(f"Spec version:  {SPEC_VERSION}")
print(f"Hypothesis:    {HYPOTHESIS}")
print(f"Seed:          {SEED}")
print(f"Baseline run:  {BASELINE_RUN or '(none)'}")
print(f"Max steps:     {MAX_STEPS:,}")
print(f"Resume:        {RESUME_TRAINING}")
print(f"Run pattern:   {RUN_ID_PATTERN}")

---
## Step 1: 依存関係と Google Drive

In [ ]:
#@title ML-Agents のインストール { run: "auto" }
!pip install -q mlagents==1.1.0 protobuf==3.20.3

import subprocess
result = subprocess.run(['mlagents-learn', '--version'], capture_output=True, text=True)
print(f"ML-Agents installed: {result.stdout.strip()}")

In [ ]:
#@title Google Drive のマウント { run: "auto" }
from google.colab import drive

drive.mount('/content/drive')

for directory in [DRIVE_BUILDS, DRIVE_RESULTS, DRIVE_MODELS]:
    os.makedirs(directory, exist_ok=True)

print(f"Builds:  {DRIVE_BUILDS}")
print(f"Results: {DRIVE_RESULTS}")
print(f"Models:  {DRIVE_MODELS}")

---
## Step 2: Unity ビルドの展開

In [ ]:
#@title ZIP を展開して構成を確認 { run: "auto" }
import glob
import shutil
import zipfile

zip_path = f"{DRIVE_BUILDS}/{SCENARIO_NAME}.zip"
if not os.path.exists(zip_path):
    available = glob.glob(f"{DRIVE_BUILDS}/*.zip")
    print(f"Missing build ZIP: {zip_path}")
    print("Available ZIP files:")
    for candidate in available:
        size_mb = os.path.getsize(candidate) / (1024 * 1024)
        print(f"  - {os.path.basename(candidate)} ({size_mb:.1f} MB)")
    raise FileNotFoundError(zip_path)

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as archive:
    archive.extractall(WORK_DIR)

exec_files = glob.glob(f"{WORK_DIR}/**/*.x86_64", recursive=True)
for exec_file in exec_files:
    os.chmod(exec_file, 0o755)

config_files = glob.glob(f"{WORK_DIR}/**/Config/*.yaml", recursive=True)
manifest_path = os.path.join(CONFIG_DIR, 'scenario_manifest.yaml')

print(f"Build ZIP: {zip_path}")
print(f"Executables: {exec_files}")
print("Config files:")
for config_file in config_files:
    print(f"  - {config_file}")
print(f"Manifest path: {manifest_path}")

if not exec_files:
    raise FileNotFoundError('No .x86_64 executable found in extracted build')

---
## Step 3: 契約検証と学習設定の確定

In [ ]:
#@title Validate Manifest and Run ID { run: "auto" }
import json
import yaml

if not HYPOTHESIS.strip():
    raise ValueError('HYPOTHESIS must not be empty')

if not re.fullmatch(RUN_ID_PATTERN, RUN_ID):
    raise ValueError(
        f"RUN_ID does not match the required pattern: {RUN_ID_PATTERN}\n"
        f"Current RUN_ID: {RUN_ID}"
    )

if not os.path.exists(manifest_path):
    raise FileNotFoundError(f"scenario_manifest.yaml not found: {manifest_path}")

with open(manifest_path, 'r', encoding='utf-8') as stream:
    manifest = yaml.safe_load(stream)

required_manifest_keys = [
    'scenario_name', 'scene_name', 'agent_class', 'behavior_name', 'learning_goal',
    'success_conditions', 'failure_conditions', 'observation_contract', 'action_contract',
    'reward_rules', 'randomization_knobs', 'difficulty_stages', 'visual_theme',
    'camera_plan', 'acceptance_criteria', 'baseline_run', 'spec_version'
]
missing_keys = [key for key in required_manifest_keys if key not in manifest]
if missing_keys:
    raise KeyError(f"manifest is missing required keys: {missing_keys}")

if manifest['scenario_name'] != SCENARIO_NAME:
    raise ValueError(f"SCENARIO_NAME ({SCENARIO_NAME}) does not match manifest scenario_name ({manifest['scenario_name']})")

if str(manifest['spec_version']) != str(SPEC_VERSION):
    raise ValueError(f"SPEC_VERSION ({SPEC_VERSION}) does not match manifest spec_version ({manifest['spec_version']})")

training_configs = [
    path for path in config_files
    if os.path.basename(path) != 'scenario_manifest.yaml'
]
if not training_configs:
    raise FileNotFoundError('No training config YAML found next to scenario_manifest.yaml')

config_path = training_configs[0]
with open(config_path, 'r', encoding='utf-8') as stream:
    config = yaml.safe_load(stream)

behavior_names = list(config.get('behaviors', {}).keys())
if manifest['behavior_name'] not in behavior_names:
    raise ValueError(
        f"manifest behavior_name ({manifest['behavior_name']}) not found in training config behaviors ({behavior_names})"
    )

for behavior_name, behavior_config in config.get('behaviors', {}).items():
    behavior_config['max_steps'] = MAX_STEPS
    print(f"{behavior_name}: max_steps -> {MAX_STEPS:,}")

with open(config_path, 'w', encoding='utf-8') as stream:
    yaml.dump(config, stream, sort_keys=False, default_flow_style=False, allow_unicode=True)

effective_baseline_run = BASELINE_RUN.strip() or str(manifest.get('baseline_run', '')).strip()
if BASELINE_RUN.strip() and str(manifest.get('baseline_run', '')).strip() and BASELINE_RUN.strip() != str(manifest.get('baseline_run', '')).strip():
    raise ValueError('BASELINE_RUN input does not match manifest baseline_run')

run_context = {
    'scenario_name': SCENARIO_NAME,
    'scenario_slug': SCENARIO_SLUG,
    'run_id': RUN_ID,
    'spec_version': SPEC_VERSION,
    'hypothesis': HYPOTHESIS,
    'seed': SEED,
    'baseline_run': effective_baseline_run,
    'max_steps': MAX_STEPS,
    'resume_training': RESUME_TRAINING,
    'zip_path': zip_path,
    'manifest_path': manifest_path,
    'config_path': config_path,
    'behavior_names': behavior_names,
}

os.makedirs(RUN_RESULTS_DIR, exist_ok=True)
with open(RUN_CONTEXT_PATH, 'w', encoding='utf-8') as stream:
    json.dump(run_context, stream, ensure_ascii=False, indent=2)

print(json.dumps(run_context, ensure_ascii=False, indent=2))
print(f"Run context saved: {RUN_CONTEXT_PATH}")

In [ ]:
#@title Run Training { run: "auto" }
import glob
import html
import subprocess
from collections import deque
from IPython.display import HTML, display

exec_path = exec_files[0]
resume_flag = '--resume' if RESUME_TRAINING else '--force'

cmd = [
    'mlagents-learn',
    config_path,
    f'--env={exec_path}',
    f'--run-id={RUN_ID}',
    f'--seed={SEED}',
    '--no-graphics',
    f'--results-dir={DRIVE_RESULTS}',
    resume_flag,
]

step_pattern = re.compile(r'Step:\s*([0-9]+)')
reward_pattern = re.compile(r'Mean Reward:\s*(-?[0-9]+(?:\.[0-9]+)?)')
elapsed_pattern = re.compile(r'Time Elapsed:\s*([0-9]+(?:\.[0-9]+)?)\s*s')
recent_logs = deque(maxlen=12)

def utc_now():
    return datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')

def collect_artifacts():
    run_dir = os.path.join(DRIVE_RESULTS, RUN_ID)
    checkpoints = glob.glob(f"{run_dir}/**/*.pt", recursive=True)
    onnx_models = glob.glob(f"{run_dir}/**/*.onnx", recursive=True)
    event_files = glob.glob(f"{run_dir}/**/events.out.tfevents.*", recursive=True)
    candidates = checkpoints + onnx_models + event_files
    latest_artifact = max(candidates, key=os.path.getmtime) if candidates else ''
    return {
        'checkpoint_count': len(checkpoints),
        'onnx_count': len(onnx_models),
        'event_file_count': len(event_files),
        'latest_artifact': os.path.basename(latest_artifact) if latest_artifact else '',
    }

progress = {
    'scenario_name': SCENARIO_NAME,
    'run_id': RUN_ID,
    'status': 'starting',
    'stage': 'training',
    'max_steps': MAX_STEPS,
    'current_step': 0,
    'mean_reward': None,
    'elapsed_seconds': 0.0,
    'checkpoint_count': 0,
    'onnx_count': 0,
    'event_file_count': 0,
    'latest_artifact': '',
    'recent_logs': [],
    'updated_utc': utc_now(),
}

def persist_progress():
    os.makedirs(RUN_RESULTS_DIR, exist_ok=True)
    progress['recent_logs'] = list(recent_logs)
    progress['updated_utc'] = utc_now()
    progress.update(collect_artifacts())
    with open(RUN_PROGRESS_PATH, 'w', encoding='utf-8') as stream:
        json.dump(progress, stream, ensure_ascii=False, indent=2)

def render_dashboard():
    percent = 0.0
    if MAX_STEPS:
        percent = min(progress['current_step'] / MAX_STEPS * 100.0, 100.0)

    reward_text = '-' if progress['mean_reward'] is None else f"{progress['mean_reward']:.3f}"
    logs_html = '<br>'.join(html.escape(line) for line in recent_logs) or 'Waiting for trainer output...'
    latest_artifact = progress['latest_artifact'] or '-'

    return f"""
    <div style='font-family:Arial,sans-serif;border:1px solid #d6dce5;border-radius:12px;padding:16px;background:#f7f9fc;'>
      <div style='display:flex;justify-content:space-between;align-items:center;gap:12px;flex-wrap:wrap;'>
        <div>
          <div style='font-size:20px;font-weight:700;color:#16324f;'>RL Movie Training Progress</div>
          <div style='font-size:13px;color:#52667a;'>Run ID: {html.escape(RUN_ID)}</div>
        </div>
        <div style='font-size:13px;color:#52667a;'>Status: <strong>{html.escape(progress['status'])}</strong></div>
      </div>
      <div style='margin-top:14px;height:14px;background:#dde6f0;border-radius:999px;overflow:hidden;'>
        <div style='width:{percent:.2f}%;height:100%;background:linear-gradient(90deg,#2e86de,#37c978);'></div>
      </div>
      <div style='margin-top:8px;font-size:12px;color:#52667a;'>{progress['current_step']:,} / {MAX_STEPS:,} steps ({percent:.1f}%)</div>
      <div style='display:grid;grid-template-columns:repeat(auto-fit,minmax(160px,1fr));gap:10px;margin-top:14px;'>
        <div style='background:white;border-radius:10px;padding:10px;'><div style='font-size:12px;color:#6b7c8f;'>Mean reward</div><div style='font-size:24px;font-weight:700;color:#16324f;'>{reward_text}</div></div>
        <div style='background:white;border-radius:10px;padding:10px;'><div style='font-size:12px;color:#6b7c8f;'>Elapsed</div><div style='font-size:24px;font-weight:700;color:#16324f;'>{progress['elapsed_seconds']:.1f}s</div></div>
        <div style='background:white;border-radius:10px;padding:10px;'><div style='font-size:12px;color:#6b7c8f;'>Checkpoints</div><div style='font-size:24px;font-weight:700;color:#16324f;'>{progress['checkpoint_count']}</div></div>
        <div style='background:white;border-radius:10px;padding:10px;'><div style='font-size:12px;color:#6b7c8f;'>Event files</div><div style='font-size:24px;font-weight:700;color:#16324f;'>{progress['event_file_count']}</div></div>
      </div>
      <div style='margin-top:14px;background:white;border-radius:10px;padding:10px;'>
        <div style='font-size:12px;color:#6b7c8f;'>Latest artifact</div>
        <div style='font-size:14px;font-weight:600;color:#16324f;'>{html.escape(latest_artifact)}</div>
      </div>
      <div style='margin-top:14px;background:#0f1720;color:#d9e2ec;border-radius:10px;padding:12px;'>
        <div style='font-size:12px;color:#8ea3b7;margin-bottom:8px;'>Recent trainer logs</div>
        <div style='font-family:Consolas,Monaco,monospace;font-size:12px;line-height:1.5;'>{logs_html}</div>
      </div>
      <div style='margin-top:10px;font-size:12px;color:#52667a;'>progress.json: {html.escape(RUN_PROGRESS_PATH)}</div>
    </div>
    """

print('Training command:')
print(' '.join(cmd))

os.makedirs(RUN_RESULTS_DIR, exist_ok=True)
persist_progress()
display_handle = display(HTML(render_dashboard()), display_id=True)

with open(RUN_LOG_PATH, 'w', encoding='utf-8') as log_stream:
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env={**os.environ, 'PYTHONUNBUFFERED': '1'},
    )

    for raw_line in process.stdout:
        line = raw_line.rstrip()
        if not line:
            continue

        log_stream.write(line + '\n')
        log_stream.flush()
        recent_logs.append(line)

        step_match = step_pattern.search(line)
        reward_match = reward_pattern.search(line)
        elapsed_match = elapsed_pattern.search(line)

        if step_match:
            progress['current_step'] = int(step_match.group(1))
        if reward_match:
            progress['mean_reward'] = float(reward_match.group(1))
        if elapsed_match:
            progress['elapsed_seconds'] = float(elapsed_match.group(1))

        progress['status'] = 'running'
        persist_progress()
        display_handle.update(HTML(render_dashboard()))

    return_code = process.wait()

progress['status'] = 'completed' if return_code == 0 else f"failed ({return_code})"
persist_progress()
display_handle.update(HTML(render_dashboard()))

if return_code != 0:
    raise RuntimeError(f"Training failed with exit code {return_code}")

print(f"Training log saved: {RUN_LOG_PATH}")

---
## Step 4: 結果確認と成果物の保存

In [ ]:
#@title View TensorBoard { run: "auto" }
%load_ext tensorboard
%tensorboard --logdir {RUN_RESULTS_DIR}

In [ ]:
#@title Export Model and Summary { run: "auto" }
import glob
import shutil

models = glob.glob(f"{DRIVE_RESULTS}/{RUN_ID}/**/*.onnx", recursive=True)
copied_models = []
os.makedirs(MODEL_EXPORT_DIR, exist_ok=True)
os.makedirs(RUN_RESULTS_DIR, exist_ok=True)

for model_path in models:
    destination = os.path.join(MODEL_EXPORT_DIR, os.path.basename(model_path))
    shutil.copy2(model_path, destination)
    copied_models.append(destination)

run_summary = {
    'scenario_name': SCENARIO_NAME,
    'scenario_slug': SCENARIO_SLUG,
    'run_id': RUN_ID,
    'spec_version': SPEC_VERSION,
    'hypothesis': HYPOTHESIS,
    'seed': SEED,
    'baseline_run': run_context['baseline_run'],
    'resume_training': RESUME_TRAINING,
    'max_steps': MAX_STEPS,
    'timestamp_utc': datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z'),
    'zip_path': zip_path,
    'config_path': config_path,
    'manifest_path': manifest_path,
    'behavior_names': behavior_names,
    'model_export_dir': MODEL_EXPORT_DIR,
    'run_context_path': RUN_CONTEXT_PATH,
    'progress_path': RUN_PROGRESS_PATH,
    'training_log_path': RUN_LOG_PATH,
    'copied_models': copied_models,
    'manifest': manifest,
}

summary_path = os.path.join(RUN_RESULTS_DIR, 'run_summary.json')
with open(summary_path, 'w', encoding='utf-8') as stream:
    json.dump(run_summary, stream, ensure_ascii=False, indent=2)

print(f"Models copied to: {MODEL_EXPORT_DIR}")
for model_path in copied_models:
    print(f"  - {model_path}")
print(f"Progress file: {RUN_PROGRESS_PATH}")
print(f"Training log: {RUN_LOG_PATH}")
print(f"Run summary: {summary_path}")

if not copied_models:
    print('Warning: no ONNX models were found in the results directory.')

In [ ]:
#@title 最新モデルをダウンロードする (任意) { run: "auto" }
from google.colab import files

if copied_models:
    latest_model = copied_models[-1]
    print(f"Downloading: {os.path.basename(latest_model)}")
    files.download(latest_model)
else:
    print('No model available for download.')